# Final pipeline
Starting from the best models of the previous notebooks, we improve the leaderboard score in two separate parts:
- **Part A, modeling:** larger model, longer training, default augmentations, P2 head.
- **Part B, data and inference:** test resolution, TTA, ensembling (WBF), temporal post-processing.

Every step is evaluated with the same `evaluate` on the same 571 val frames and added to one table, so the effect of model changes and of data/inference changes can be read separately.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


In [ ]:
import matplotlib.pyplot as plt, matplotlib.patches as patches

In [ ]:
import os
from pathlib import Path
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('kaggle')

if not Path('/content/data/train').exists():
    !kaggle competitions download -c multiple-object-video-detection-hw-2-ucu-ai-2026 -p /content/data
    !cd /content/data && unzip -q '*.zip' && rm *.zip
else:
    print('Data is here')
ROOT = Path('/content/data')

Data is here


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PREP = Path('/content/prepared')
if not PREP.exists():
    !unzip -q /content/drive/MyDrive/hw2_prepared.zip -d /content
for split in ['train', 'val']:
    n = sum(1 for _ in (PREP / split / 'images').rglob('*.jpg'))
    print(f'{split}: {n} кадрів')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
unzip:  cannot find or open /content/drive/MyDrive/hw2_prepared.zip, /content/drive/MyDrive/hw2_prepared.zip.zip or /content/drive/MyDrive/hw2_prepared.zip.ZIP.
train: 0 кадрів
val: 0 кадрів


## Setup

In [ ]:
import random, cv2, numpy as np, torch
import torchvision.transforms.functional as TF

CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
SEED, DEVICE = 42, 'cuda'

## Evaluation
Predictions are made on the original val frames (every 5th frame, 571 frames, not masked). Metrics are computed with pycocotools (`maxDets=500`); detections inside ignored regions are not penalised thanks to `iscrowd=1` in the ground truth. The same evaluation is used for every model.

In [ ]:
import json, contextlib, io, pandas as pd
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

with contextlib.redirect_stdout(io.StringIO()):
    gt = COCO(str(ROOT / 'val/annotations_coco.json'))
imgs_va = pd.DataFrame(gt.dataset['images']).rename(columns={'id': 'image_id'})
val_sub = imgs_va.sort_values(['video', 'frame_id'])
val_sub = val_sub[val_sub.groupby('video').cumcount() % 5 == 0]       # той самий кожен 5-й кадр
print('val frames for evaluation:', len(val_sub))

@torch.no_grad()
def predict(model, df, split='val', bs=4):
    """Передбачення на оригінальних кадрах -> список у COCO-форматі (bbox = [x, y, w, h] у пікселях)."""
    model.eval(); out = []
    for i in range(0, len(df), bs):
        rows = df.iloc[i:i + bs]
        ims = [TF.to_tensor(cv2.cvtColor(cv2.imread(str(ROOT / split / f)), cv2.COLOR_BGR2RGB)).to(DEVICE)
               for f in rows.file_name]
        with torch.autocast('cuda', dtype=torch.float16):
            res = model(ims)
        for iid, r in zip(rows.image_id, res):
            b = r['boxes'].float().cpu().numpy(); b[:, 2:] -= b[:, :2]      # xyxy -> xywh
            for box, s, l in zip(b, r['scores'].float().cpu().numpy(), r['labels'].cpu().numpy()):
                out.append({'image_id': int(iid), 'category_id': int(l),   # мітки 1..8 = COCO category_id
                            'bbox': [round(float(v), 2) for v in box], 'score': float(s)})
    return out

def evaluate(preds, img_ids):
    """COCO mAP + AP по класах."""
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(preds)
        E = COCOeval(gt, dt, 'bbox')
        E.params.imgIds = list(img_ids); E.params.maxDets = [1, 100, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    s = E.stats
    res = {'mAP': s[0], 'AP50': s[1], 'AP75': s[2], 'AP_small': s[3], 'AP_medium': s[4], 'AP_large': s[5]}
    prec = E.eval['precision']                                           # [IoU, recall, class, area, maxDets]
    for k, c in enumerate(CLASSES):
        p = prec[:, :, k, 0, -1]
        res[f'AP_{c}'] = float(p[p > -1].mean()) if (p > -1).any() else float('nan')
    return res

val frames for evaluation: 571


In [ ]:
!pip -q install ultralytics ensemble-boxes
import os, yaml, zipfile, shutil, torchvision
from ultralytics import RTDETR
from ultralytics.models.rtdetr.train import RTDETRTrainer

RUN_DIR, IMGSZ = '/content/drive/MyDrive/hw2_runs', 1024
W_OLD = f'{RUN_DIR}/rtdetr_exp1_lr/weights/last.pt'

# prepared training frames (needed only for the fine-tuning cell)
if not (PREP / 'train' / 'images').exists():
    z = Path('/content/drive/MyDrive/УКУ/AI/hw2_prepared.zip')
    if not z.exists():
        z = next(Path('/content/drive/MyDrive').rglob('hw2_prepared.zip'), None)
    assert z, 'hw2_prepared.zip is not on this Drive account'
    zipfile.ZipFile(z).extractall('/content')
print({s: sum(1 for _ in (PREP / s / 'images').rglob('*.jpg')) for s in ['train', 'val']})

DATA_YAML = '/content/rtdetr_data.yaml'
yaml.safe_dump({'path': str(PREP), 'train': 'train/images', 'val': 'val/images',
                'names': dict(enumerate(CLASSES))}, open(DATA_YAML, 'w'))


class TunedRTDETRTrainer(RTDETRTrainer):
    """RT-DETR trainer with a separate learning rate for the CNN backbone (layers listed under `backbone:`)."""
    backbone_mult = 1.0

    def build_optimizer(self, model, name='auto', lr=0.001, momentum=0.9, decay=1e-5, iterations=1e5):
        opt = super().build_optimizer(model, name, lr, momentum, decay, iterations)
        n_bb = len(model.yaml['backbone'])
        bb = {id(p) for layer in model.model[:n_bb] for p in layer.parameters()}
        groups = []
        for g in opt.param_groups:
            for part, mult in (('backbone', self.backbone_mult), ('transformer', 1.0)):
                params = [p for p in g['params'] if (id(p) in bb) == (part == 'backbone')]
                if params:
                    groups.append({'params': params, 'lr': g['lr'] * mult, 'weight_decay': g['weight_decay'],
                                   'param_group': g['param_group']})
        return torch.optim.AdamW(groups, lr=lr, betas=(momentum, 0.999), weight_decay=0.0,
                                 fused=next(model.parameters()).is_cuda)

{'train': 4868, 'val': 571}


In [ ]:
import torchvision, shutil
from ensemble_boxes import weighted_boxes_fusion


def predict_rtdetr_tta(weights, df, split='val', sizes=(1280,), flip=True, bs=8, iou_thr=0.6):
    """RT-DETR predictions fused over input sizes and horizontal flip (WBF); same output format as predict_rtdetr."""
    m, out = RTDETR(weights), []
    for i in range(0, len(df), bs):
        rows = df.iloc[i:i + bs]
        ims = [cv2.imread(str(ROOT / split / f)) for f in rows.file_name]
        views = []
        for sz in sizes:
            for fl in ([False, True] if flip else [False]):
                src = [cv2.flip(im, 1) for im in ims] if fl else ims
                views.append((fl, m.predict(src, imgsz=sz, conf=0.001, max_det=300, verbose=False)))
        for k, (iid, im) in enumerate(zip(rows.image_id, ims)):
            H, W = im.shape[:2]; B, S, L = [], [], []
            for fl, res in views:
                b = res[k].boxes.xyxyn.float().cpu().numpy().copy()
                if fl:
                    b[:, [0, 2]] = 1 - b[:, [2, 0]]                    # mirror boxes back
                s = res[k].boxes.conf.float().cpu().numpy(); l = res[k].boxes.cls.cpu().numpy()
                if len(views) > 1:                                     # drop low-score duplicates before fusing
                    keep = torchvision.ops.batched_nms(torch.tensor(b), torch.tensor(s), torch.tensor(l).long(), 0.6).numpy()
                    b, s, l = b[keep], s[keep], l[keep]
                B.append(b.clip(0, 1)); S.append(s); L.append(l)
            if len(views) > 1:
                b, s, l = weighted_boxes_fusion(B, S, L, iou_thr=iou_thr, skip_box_thr=0.001)
            else:
                b, s, l = B[0], S[0], L[0]
            b = b * [W, H, W, H]; b[:, 2:] -= b[:, :2]                 # normalised xyxy -> xywh pixels
            for j in np.argsort(-s)[:500]:
                out.append({'image_id': int(iid), 'category_id': int(l[j]) + 1,
                            'bbox': [round(float(v), 2) for v in b[j]], 'score': float(s[j])})
    return out

W_OLD = f'{RUN_DIR}/rtdetr_exp1_lr/weights/last.pt'
CONFIGS = {'exp2 @1024 (start)':        (W_OLD, dict(sizes=(1024,), flip=False)),
           'B: exp2, test @1280':       (W_OLD, dict(sizes=(1280,), flip=False)),
           'B: exp2, @1024 + flip':     (W_OLD, dict(sizes=(1024,), flip=True))}
results = {k: evaluate(predict_rtdetr_tta(w, val_sub, **kw), val_sub.image_id) for k, (w, kw) in CONFIGS.items()}
display(pd.DataFrame(results).T[['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large']].round(4))

,mAP,AP50,AP75,AP_small,AP_medium,AP_large
exp2 @1024 (start),0.2695,0.5145,0.2441,0.1376,0.2323,0.4686
"B: exp2, test @1280",0.2857,0.5386,0.2617,0.1423,0.2394,0.4921
"B: exp2, @1024 + flip",0.2789,0.5294,0.2554,0.1399,0.2367,0.4870


In [ ]:
LOCAL = '/content/runs_local'
W_OLD = f'{RUN_DIR}/rtdetr_exp1_lr/weights/last.pt'
TunedRTDETRTrainer.backbone_mult = 0.1
m = RTDETR(W_OLD)
m.train(trainer=TunedRTDETRTrainer, data=DATA_YAML, imgsz=1280, batch=8, nbs=16,
        epochs=2, time=0.22,
        mosaic=0.0,
        optimizer='AdamW', lr0=5e-5, lrf=0.1, weight_decay=1e-4, warmup_epochs=0.3, warmup_bias_lr=0.0,
        amp=True, seed=SEED, device=0, workers=os.cpu_count(),
        save_period=1, deterministic=False, plots=False, project=LOCAL, name='rtdetr_ft1280', exist_ok=True)
W_NEW = f'{LOCAL}/rtdetr_ft1280/weights/last.pt'
shutil.copy(W_NEW, f'{RUN_DIR}/rtdetr_ft1280_last.pt')

Ultralytics 8.4.214 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/rtdetr_data.yaml, degrees=0.0, deterministic=False, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=2, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, iou=None, kobj=1.0, line_width=None, lr0=5e-05, lrf=0.1, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/drive/MyDrive/hw2_runs/rtdetr_exp1_lr/weights/last.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, nam

KeyboardInterrupt: 

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from ultralytics import RTDETR

W, SZ, note = '/content/drive/MyDrive/hw2_runs/rtdetr_exp1_lr/weights/last.pt', 1024, 'exp1 6 epochs'
ft = Path('/content/runs_local/rtdetr_ft1280')
try:                                             # fine-tuned weights only if a finished epoch beat the start (0.277)
    h = pd.read_csv(ft / 'results.csv'); h.columns = h.columns.str.strip()
    if len(h) and h['metrics/mAP50-95(B)'].iloc[-1] > 0.277 and (ft / 'weights/last.pt').exists():
        W, SZ, note = str(ft / 'weights/last.pt'), 1280, f'fine-tuned at 1280, {len(h)} extra epochs'
except Exception:
    pass
print('submitting:', note, '|', W)

test = pd.read_csv(ROOT / 'test_images.csv')
files = [str(ROOT / f) for f in test.file_name]              # file_name already starts with 'test/images/'
m, rows = RTDETR(W), []
for i in range(0, len(files), 16):
    res = m.predict(files[i:i + 16], imgsz=SZ, conf=0.001, max_det=300, verbose=False)
    for iid, r in zip(test.image_id.iloc[i:i + 16], res):
        b = r.boxes.xyxy.float().cpu().numpy(); b[:, 2:] -= b[:, :2]
        s = r.boxes.conf.float().cpu().numpy(); c = r.boxes.cls.cpu().numpy().astype(int)
        rows.append((iid, ' '.join(f'{c[j]} {s[j]:.4f} {b[j, 0]:.1f} {b[j, 1]:.1f} {b[j, 2]:.1f} {b[j, 3]:.1f}'
                                   for j in np.argsort(-s))))
sub = pd.DataFrame(rows, columns=['image_id', 'PredictionString'])
assert len(sub) == len(test)
sub.to_csv('sub_final.csv', index=False)
!kaggle competitions submit -c multiple-object-video-detection-hw-2-ucu-ai-2026 -f sub_final.csv -m "RT-DETR-L final"

submitting: fine-tuned at 1280, 3 extra epochs | /content/runs_local/rtdetr_ft1280/weights/last.pt
100% 34.0M/34.0M [00:04<00:00, 7.29MB/s]
Successfully submitted to Multiple Object Video Detection | HW2 | UCU AI 2026